# 03 — Build Event Universe

Applies the **pre-registered inclusion rule** to `data/markets_metadata.csv`
and writes `data/event_universe.csv`.

**Inclusion rule (fixed before seeing results):**
- Mutually-exclusive single-winner event (exactly one YES sub-market)
- ≥ 4 sub-markets
- ≥ 500 total trades across all sub-markets
- All sub-markets settled (status = finalized)
- Not a KXMVE noise market

**STOP after this notebook** — review the event table before running 04.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

BASE_DIR  = os.path.abspath('..')
DATA_DIR  = os.path.join(BASE_DIR, 'data')
OUT_DIR   = os.path.join(BASE_DIR, 'output')
META_PATH = os.path.join(DATA_DIR, 'markets_metadata.csv')
OUT_PATH  = os.path.join(DATA_DIR, 'event_universe.csv')

os.makedirs(OUT_DIR, exist_ok=True)
print(f'DATA_DIR  : {DATA_DIR}')
print(f'META_PATH : {META_PATH}')
print(f'OUT_PATH  : {OUT_PATH}')

DATA_DIR  : /Users/josephsemelroth/Desktop/QSS 20 Final Project/data
META_PATH : /Users/josephsemelroth/Desktop/QSS 20 Final Project/data/markets_metadata.csv
OUT_PATH  : /Users/josephsemelroth/Desktop/QSS 20 Final Project/data/event_universe.csv


In [2]:
meta = pd.read_csv(META_PATH, low_memory=False)
print(f'Loaded {len(meta):,} rows from markets_metadata.csv')
print(f'Columns: {meta.columns.tolist()}')
print()
print(f'Unique event_tickers: {meta["event_ticker"].nunique():,}')
print()
print('status counts:')
print(meta['status'].value_counts(dropna=False).to_string())
print()
print('result counts:')
print(meta['result'].value_counts(dropna=False).to_string())
print()
print('analysis_category counts:')
print(meta['analysis_category'].value_counts(dropna=False).to_string())

Loaded 3,393 rows from markets_metadata.csv
Columns: ['ticker', 'event_ticker', 'title', 'open_time', 'close_time', 'settlement_ts', 'volume_fp', 'result', 'status', 'analysis_category']

Unique event_tickers: 488

status counts:
status
finalized    3393

result counts:
result
no        2710
yes        653
scalar      30

analysis_category counts:
analysis_category
Political    1971
Sports       1422


In [3]:
meta['volume_fp'] = pd.to_numeric(meta['volume_fp'], errors='coerce').fillna(0)

ev = (
    meta.groupby('event_ticker', sort=False)
    .agg(
        n_sub_markets            = ('ticker',            'count'),
        total_trades             = ('volume_fp',         'sum'),
        category                 = ('analysis_category', 'first'),
        n_finalized              = ('status',            lambda x: (x == 'finalized').sum()),
        n_yes                    = ('result',            lambda x: (x == 'yes').sum()),
        resolution_or_close_time = ('settlement_ts',     'max'),
    )
    .reset_index()
)

print(f'Events (raw, before any filter): {len(ev):,}')
print()
print('n_sub_markets distribution (all events):')
print(ev['n_sub_markets'].describe().apply(lambda x: f'{x:.1f}').to_string())
print()
print('total_trades distribution (all events):')
print(ev['total_trades'].describe().apply(lambda x: f'{x:,.0f}').to_string())
print()
print('n_yes distribution (all events):')
print(ev['n_yes'].value_counts().head(10).to_string())

Events (raw, before any filter): 488

n_sub_markets distribution (all events):
count    488.0
mean       7.0
std        9.7
min        1.0
25%        2.0
50%        4.0
75%        8.0
max      137.0

total_trades distribution (all events):
count            488
mean       1,480,782
std       12,153,521
min                0
25%            3,342
50%           18,514
75%          135,260
max      216,970,762

n_yes distribution (all events):
n_yes
1     325
0     102
2      14
5      11
3      11
4       9
6       5
10      3
14      2
11      1


In [4]:
print('PRE-REGISTERED INCLUSION RULE (thresholds fixed before seeing results)')
print('  1. Not a KXMVE noise market')
print('  2. All sub-markets settled (all finalized)')
print('  3. Exactly one YES winner (mutually-exclusive single-winner event)')
print('  4. >= 4 sub-markets')
print('  5. >= 500 total trades across all sub-markets')
print()

n0 = len(ev)

step1 = ev[~ev['event_ticker'].str.startswith('KXMVE', na=False)].copy()
print(f'Step 1 — exclude KXMVE:               {len(step1):>5}  (removed {n0 - len(step1):>4})')

step2 = step1[step1['n_finalized'] == step1['n_sub_markets']].copy()
print(f'Step 2 — all sub-markets finalized:   {len(step2):>5}  (removed {len(step1) - len(step2):>4})')

step3 = step2[step2['n_yes'] == 1].copy()
print(f'Step 3 — exactly 1 YES winner:        {len(step3):>5}  (removed {len(step2) - len(step3):>4})')

step4 = step3[step3['n_sub_markets'] >= 4].copy()
print(f'Step 4 — >= 4 sub-markets:            {len(step4):>5}  (removed {len(step3) - len(step4):>4})')

step5 = step4[step4['total_trades'] >= 500].copy()
print(f'Step 5 — >= 500 total trades:         {len(step5):>5}  (removed {len(step4) - len(step5):>4})')

# Step 6 (added per pre-analysis decision): exclude KXTRUTHSOCIAL repeated weekly markets.
# Rationale: KXTRUTHSOCIAL-26FEB28/-MAR07/-MAR14/-MAR21 are the same market format
# repeated on a weekly schedule -- not independent events.
step6 = step5[~step5['event_ticker'].str.startswith('KXTRUTHSOCIAL', na=False)].copy()
print(f'Step 6 — exclude KXTRUTHSOCIAL weekly: {len(step6):>5}  (removed {len(step5) - len(step6):>4})')
print()
print(f'FINAL EVENT UNIVERSE: {len(step6)} events')

PRE-REGISTERED INCLUSION RULE (thresholds fixed before seeing results)
  1. Not a KXMVE noise market
  2. All sub-markets settled (all finalized)
  3. Exactly one YES winner (mutually-exclusive single-winner event)
  4. >= 4 sub-markets
  5. >= 500 total trades across all sub-markets

Step 1 — exclude KXMVE:                 488  (removed    0)
Step 2 — all sub-markets finalized:     488  (removed    0)
Step 3 — exactly 1 YES winner:          325  (removed  163)
Step 4 — >= 4 sub-markets:              187  (removed  138)
Step 5 — >= 500 total trades:           187  (removed    0)
Step 6 — exclude KXTRUTHSOCIAL weekly:   182  (removed    5)

FINAL EVENT UNIVERSE: 182 events


In [5]:
def _domain(cat):
    if cat == 'Sports':    return 'Sports'
    if cat == 'Political': return 'Political'
    return 'Other'

step6 = step6.copy()
step6['domain'] = step6['category'].apply(_domain)

event_universe = (
    step6[['event_ticker', 'category', 'domain', 'n_sub_markets',
           'total_trades', 'resolution_or_close_time']]
    .sort_values('event_ticker')
    .reset_index(drop=True)
)

print(f'Row count: {len(event_universe):,}')
event_universe.to_csv(OUT_PATH, index=False)
print(f'Saved {len(event_universe):,} rows -> {OUT_PATH}')
print()
print(event_universe.dtypes.to_string())

Row count: 182
Saved 182 rows -> /Users/josephsemelroth/Desktop/QSS 20 Final Project/data/event_universe.csv

event_ticker                 object
category                     object
domain                       object
n_sub_markets                 int64
total_trades                float64
resolution_or_close_time     object


In [6]:
print('=' * 62)
print(f'TOTAL EVENTS IN UNIVERSE: {len(event_universe)}')
print('=' * 62)
print()
print('By domain:')
for dom, ct in event_universe['domain'].value_counts().items():
    print(f'  {dom:<12}: {ct:>4}  ({100*ct/len(event_universe):.1f}%)')
print()
print('By category:')
for cat, ct in event_universe['category'].value_counts().items():
    print(f'  {cat:<12}: {ct:>4}  ({100*ct/len(event_universe):.1f}%)')
print()
print('n_sub_markets statistics:')
print(event_universe['n_sub_markets'].describe().apply(lambda x: f'{x:.1f}').to_string())
print()
print('total_trades statistics (by domain):')
print(event_universe.groupby('domain')['total_trades'].describe()
      .applymap(lambda x: f'{x:,.0f}').to_string())
print()
print('Full event table:')
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 120)
print(event_universe[['event_ticker', 'domain', 'n_sub_markets', 'total_trades']]
      .assign(total_trades=event_universe['total_trades'].apply(lambda x: f'{x:,.0f}'))
      .to_string(index=True))

TOTAL EVENTS IN UNIVERSE: 182

By domain:
  Political   :  137  (75.3%)
  Sports      :   45  (24.7%)

By category:
  Political   :  137  (75.3%)
  Sports      :   45  (24.7%)

n_sub_markets statistics:
count    182.0
mean      12.4
std       13.2
min        4.0
25%        5.0
50%        8.0
75%       13.0
max      137.0

total_trades statistics (by domain):
          count       mean         std    min     25%      50%        75%          max
domain                                                                                
Political   137  2,678,287  18,669,047    714  11,018   59,266    379,491  216,970,762
Sports       45  4,817,506  21,436,742  7,911  36,340  145,680  1,272,031  142,086,661

Full event table:
                          event_ticker     domain  n_sub_markets total_trades
0               KXACAHOUSEVOTE-27JAN01  Political             10       92,418
1                    KXALCARAZCOACH-26     Sports             12       36,340
2      KXARGENTINAHOUSETHIRD-25OCT26-3

/tmp/claude-501/ipykernel_58500/3596974747.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(lambda x: f'{x:,.0f}').to_string())
